# GéoMarketing IDF : J10 — Revenus et niveau de vie des communes franciliennes

## Objectifs

- charger les indicateurs communaux Filosofi 2023 ;
- conserver les statuts de confidentialité et de disponibilité ;
- contrôler les codes géographiques ;
- joindre les revenus au profil communal J9 ;
- comparer les communes à la référence régionale officielle ;
- enregistrer le profil communal J10.

## Indicateurs retenus

- niveau de vie médian annuel ;
- taux de pauvreté au seuil de 60 % ;
- indice de niveau de vie médian, référence IDF = 100 ;
- écart du taux de pauvreté avec l'IDF.

## Principes

- une valeur confidentielle reste manquante ;
- les médianes communales ne sont pas additionnées ;
- les taux de pauvreté ne sont pas moyennés arbitrairement ;
- aucune estimation de chiffre d'affaires n'est produite.

In [1]:
#Importation des librairies

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from zipfile import ZipFile
from datetime import datetime, timezone

import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 120)

print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)

Pandas : 2.2.2
NumPy : 1.26.4


In [11]:
#Définir les chemins

RACINE = Path.cwd()

if not (RACINE / "data").is_dir():
    if (RACINE.parent / "data").is_dir():
        RACINE = RACINE.parent
    else:
        raise FileNotFoundError(
            "Ouvre le notebook depuis GeoMarketing_IDF "
            "ou depuis son dossier notebooks."
        )

DOSSIER_RAW = (
    RACINE / "/Users/almou/OneDrive/GeoMarketing_IDF/data" / "raw" / "insee" / "filosofi2023"
)

DOSSIER_INTERIM = (
    RACINE / "/Users/almou/OneDrive/GeoMarketing_IDF/data" / "interim" / "j10"
)

DOSSIER_PROCESSED = (
    RACINE / "/Users/almou/OneDrive/GeoMarketing_IDF/data" / "processed"
)

DOSSIER_RAW.mkdir(parents=True, exist_ok=True)
DOSSIER_INTERIM.mkdir(parents=True, exist_ok=True)
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

FICHIER_SOURCE = DOSSIER_RAW / "FILOSOFI_CC_csv.zip"

FICHIER_PROFIL_J9 = (
    DOSSIER_PROCESSED / "profil_communes_idf_j9.csv"
)

FICHIER_PROFIL_J10 = (
    DOSSIER_PROCESSED / "profil_communes_idf_j10.csv"
)

for fichier in [FICHIER_SOURCE, FICHIER_PROFIL_J9]:
    if not fichier.is_file():
        raise FileNotFoundError(
            f"Fichier introuvable : {fichier}"
        )

print("Source :", FICHIER_SOURCE)
print("Entrée :", FICHIER_PROFIL_J9)
print("Sortie :", FICHIER_PROFIL_J10)

Source : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\insee\filosofi2023\FILOSOFI_CC_csv.zip
Entrée : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j9.csv
Sortie : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j10.csv


In [12]:
def normaliser_nom_colonne(nom):
    texte = unicodedata.normalize(
        "NFKD",
        str(nom).strip().upper(),
    )

    texte = "".join(
        caractere
        for caractere in texte
        if not unicodedata.combining(caractere)
    )

    return re.sub(
        r"[^A-Z0-9]+",
        "_",
        texte,
    ).strip("_")


def normaliser_code_commune(serie):
    texte = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    masque = texte.str.fullmatch(
        r"\d{1,5}",
        na=False,
    )

    return texte.where(masque).str.zfill(5)


def enregistrer_csv(table, fichier):
    table.to_csv(
        fichier,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"{fichier.name} : "
        f"{len(table):,} lignes, "
        f"{len(table.columns)} colonnes"
    )


def empreinte_sha256(fichier):
    calcul = hashlib.sha256()

    with open(fichier, "rb") as flux:
        for bloc in iter(
            lambda: flux.read(1024 * 1024),
            b"",
        ):
            calcul.update(bloc)

    return calcul.hexdigest()

In [13]:
#Charger le profil J9

profil_j9 = pd.read_csv(
    FICHIER_PROFIL_J9,
    dtype={"CODGEO": "string"},
    encoding="utf-8-sig",
    low_memory=False,
)

profil_j9.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil_j9.columns
]

if profil_j9.columns.duplicated().any():
    raise ValueError(
        "Des colonnes du profil portent le même nom "
        "après normalisation."
    )

if "CODGEO" not in profil_j9.columns:
    raise ValueError("La colonne CODGEO est absente.")

profil_j9["CODGEO"] = normaliser_code_commune(
    profil_j9["CODGEO"]
)

if profil_j9["CODGEO"].isna().any():
    raise ValueError(
        "Des codes communaux du profil sont invalides."
    )

if not profil_j9["CODGEO"].is_unique:
    raise ValueError(
        "Le profil contient plusieurs lignes par commune."
    )

codes_profil = set(profil_j9["CODGEO"])

print("Communes :", len(profil_j9))
print("Colonnes :", len(profil_j9.columns))

display(profil_j9.head())

Communes : 1266
Colonnes : 403


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,POP_18_24,POP_25_39,POP_40_54,POP_55_64,POP_65_79,POP_80_PLUS,POP_FEMMES,POP_HOMMES,POP_F_0_2,POP_F_3_5,...,PART_ACTIFS_OCCUPES_RESIDENTS_ARTISANS_COMMERCANTS_CHEFS_PCT,PART_EMPLOIS_LT_CADRES_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_CADRES_PCT,PART_EMPLOIS_LT_PROF_INTERMEDIAIRES_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_PROF_INTERMEDIAIRES_PCT,PART_EMPLOIS_LT_EMPLOYES_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_EMPLOYES_PCT,PART_EMPLOIS_LT_OUVRIERS_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_OUVRIERS_PCT,PART_EMPLOIS_LT_AGRICULTURE_PCT,PART_EMPLOIS_LT_INDUSTRIE_PCT,PART_EMPLOIS_LT_CONSTRUCTION_PCT,PART_EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES_PCT,PART_EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL_PCT,EMPLOIS_TERTIAIRES_LT,PART_EMPLOIS_TERTIAIRES_LT_PCT,EMPLOIS_15P_LT_POUR_1000_HAB,INDICE_DENSITE_EMPLOIS_IDF_BASE100,NB_RESTAURATION_RAPIDE_1000_EMPLOIS,INDICE_DENSITE_RAPIDE_PAR_EMPLOI_IDF_BASE100,NB_RESTAURANTS_TOTAL_1000_EMPLOIS,NB_LIEUX_TRANSPORT,NB_LIEUX_BUS,NB_STATIONS_METRO,NB_STATIONS_TRAMWAY,NB_GARES_RER,NB_GARES_TRAIN,NB_STATIONS_FUNICULAIRE,NB_LIEUX_FERRES,NB_GARES_STATIONS_TRANSPORT_LOURD,NB_LIEUX_MULTIMODAUX,NB_POLES_MULTIMODAUX_LOURDS,NB_LIGNES_TRANSPORT,NB_MODES_TRANSPORT_PRESENTS,NB_OPERATEURS_TRANSPORT,NB_LIGNES_BUS,NB_LIGNES_METRO,NB_LIGNES_TRAMWAY,NB_LIGNES_RER,NB_LIGNES_TRAIN,NB_LIGNES_FUNICULAIRE,NB_LIGNES_AUTRES,NB_LIGNES_FERREES,NB_LIGNES_TRANSPORT_LOURD,A_BUS,A_METRO,A_TRAMWAY,A_RER,A_TRAIN,A_TRANSPORT_LOURD,NB_LIEUX_TRANSPORT_10000_HAB,NB_LIGNES_TRANSPORT_10000_HAB,NB_LIGNES_FERREES_10000_HAB,NB_GARES_STATIONS_LOURDES_10000_HAB,NB_POLES_MULTIMODAUX_10000_HAB,INDICE_LIEUX_TRANSPORT_IDF_BASE100,INDICE_LIGNES_FERREES_IDF_BASE100,NB_LIGNES_TRANSPORT_1000_EMPLOIS,NB_GARES_STATIONS_LOURDES_1000_EMPLOIS,CLASSE_DESSERTE_TRANSPORT
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508,513386.756029,457511.658564,387099.215350,302819.494109,154040.332162,24566.982278,787667.317537,970898.414593,481426.808549,178607.314440,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.50,-0.59,33650.0,16.8,1.551134e+06,188002.0,73.384590,88.944294,20911,8329,12553,29,0,98.930551,39.404742,59.388609,39.830711,60.030606,200.612151,54155.23178,51427.58154,87388.40827,75287.99165,59522.89891,236306.98805,540939.72777,390054.10537,236685.88601,263552.66604,108456.51460,1.115440e+06,988338.37118,26522.40142,25422.98841,...,5.408460,41.777533,52.543672,23.155509,20.785441,21.451112,15.952686,8.386582,5.281306,0.050486,3.653341,3.241342,69.680190,23.374642,1.685340e+06,93.054832,860.799646,177.970034,4.599291,114.005364,11.547097,1298,1181,249,60,12,11,2,305,261,208,182,251,7,33,199,19,4,6,17,1,5,47,42,1,1,1,1,1,1,6.140876,1.187488,0.222358,1.234799,0.861047,40.394200,50.887449,0.138603,0.144125,POLE_MULTIMODAL_LOURD
1,77001,Achères-la-Forêt,77,11,1232.0,1139.0,1183.0,182.135095,159.522993,189.869837,347.331906,204.113784,89.442063,10.584321,341.658088,349.392830,304.140169,100.026384,15.40,13.48,16.05,28.88,29.53,25.71,8.46,44.0,3.86,0.63,

In [14]:
#Inspecter l'archive

with ZipFile(FICHIER_SOURCE) as archive:
    membres = archive.namelist()

print("Contenu de l'archive :")
print(membres)

candidats_data = [
    membre
    for membre in membres
    if Path(membre).name == "DS_FILOSOFI_CC_2023_data.csv"
]

candidats_metadata = [
    membre
    for membre in membres
    if Path(membre).name == "DS_FILOSOFI_CC_2023_metadata.csv"
]

if len(candidats_data) != 1 or len(candidats_metadata) != 1:
    raise ValueError(
        "Structure de l'archive différente de celle attendue. "
        "Vérifie le fichier téléchargé."
    )

MEMBRE_DATA = candidats_data[0]
MEMBRE_METADATA = candidats_metadata[0]

with ZipFile(FICHIER_SOURCE) as archive:
    with archive.open(MEMBRE_DATA) as flux:
        apercu = pd.read_csv(
            flux,
            sep=";",
            dtype="string",
            encoding="utf-8-sig",
            keep_default_na=False,
            nrows=5,
        )

print("Colonnes :")
print(apercu.columns.tolist())

display(apercu)

Contenu de l'archive :
['DS_FILOSOFI_CC_2023_data.csv', 'DS_FILOSOFI_CC_2023_metadata.csv']
Colonnes :
['GEO', 'GEO_OBJECT', 'FILOSOFI_MEASURE', 'UNIT_MULT', 'UNIT_MEASURE', 'CONF_STATUS', 'OBS_STATUS', 'TIME_PERIOD', 'OBS_VALUE']


,GEO,GEO_OBJECT,FILOSOFI_MEASURE,UNIT_MULT,UNIT_MEASURE,CONF_STATUS,OBS_STATUS,TIME_PERIOD,OBS_VALUE
0,097,AAV2020,GI_SL,0,NR,F,O,2023,
1,483,AAV2020,GI_SL,0,NR,F,O,2023,
2,249,AAV2020,GI_SL,0,NR,F,O,2023,
3,580,AAV2020,GI_SL,0,NR,F,O,2023,
4,471,AAV2020,GI_SL,0,NR,F,O,2023,


In [15]:
#Lire les métadonnées

metadata_source = pd.read_csv(
            "C:/Users/almou/OneDrive/GeoMarketing_IDF/data/raw/insee/filosofi2023/DS_FILOSOFI_CC_2023_metadata.csv",
            sep=";",
            dtype="string",
            encoding="utf-8-sig",
            keep_default_na=False,
        )

variables_a_examiner = [
    "CONF_STATUS",
    "OBS_STATUS",
    "UNIT_MEASURE",
]

display(
    metadata_source.loc[
        metadata_source["COD_VAR"].isin(
            variables_a_examiner
        )
    ]
)

,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD,GEO_OBJECT
0,CONF_STATUS,Statut de confidentialité,F,Libre (libre pour publication),
1,CONF_STATUS,Statut de confidentialité,C,Information statistique confidentielle,
29,OBS_STATUS,Statut de l'observation,O,Valeur manquante (vm),
30,OBS_STATUS,Statut de l'observation,A,Normale,
32,UNIT_MEASURE,Unité de mesure,EUR_YR,Euros par an,
33,UNIT_MEASURE,Unité de mesure,NR,Nombre,
34,UNIT_MEASURE,Unité de mesure,PT,Pourcentage,


In [16]:
#Extraire les communes IDF

DEPARTEMENTS_IDF = {
    "75", "77", "78", "91",
    "92", "93", "94", "95",
}

MESURES_RETENUES = {
    "MED_SL",
    "PR_MD60",
}

colonnes_requises = {
    "GEO",
    "GEO_OBJECT",
    "FILOSOFI_MEASURE",
    "UNIT_MULT",
    "UNIT_MEASURE",
    "CONF_STATUS",
    "OBS_STATUS",
    "TIME_PERIOD",
    "OBS_VALUE",
}

absentes = colonnes_requises - set(apercu.columns)

if absentes:
    raise ValueError(
        f"Colonnes absentes : {sorted(absentes)}"
    )

blocs_retenus = []
nb_lignes_lues = 0

with ZipFile(FICHIER_SOURCE) as archive:
    with archive.open(MEMBRE_DATA) as flux:
        with pd.read_csv(
            flux,
            sep=";",
            dtype="string",
            encoding="utf-8-sig",
            keep_default_na=False,
            chunksize=50_000,
        ) as lecteur:

            for numero_bloc, bloc in enumerate(
                lecteur,
                start=1,
            ):
                for colonne in bloc.columns:
                    bloc[colonne] = bloc[colonne].str.strip()

                nb_lignes_lues += len(bloc)

                masque_mesure = (
                    bloc["TIME_PERIOD"].eq("2023")
                    & bloc["FILOSOFI_MEASURE"].isin(
                        MESURES_RETENUES
                    )
                )

                masque_communes = (
                    bloc["GEO_OBJECT"].eq("COM")
                    & bloc["GEO"].str.fullmatch(
                        r"\d{5}",
                        na=False,
                    )
                    & bloc["GEO"].str[:2].isin(
                        DEPARTEMENTS_IDF
                    )
                )

                masque_region = (
                    bloc["GEO_OBJECT"].eq("REG")
                    & bloc["GEO"].eq("11")
                )

                extrait = bloc.loc[
                    masque_mesure
                    & (masque_communes | masque_region)
                ].copy()

                if not extrait.empty:
                    blocs_retenus.append(extrait)

                if numero_bloc % 10 == 0:
                    print(
                        f"{nb_lignes_lues:,} lignes lues"
                    )

if not blocs_retenus:
    raise ValueError(
        "Aucune donnée correspondant aux filtres."
    )

revenus_long = pd.concat(
    blocs_retenus,
    ignore_index=True,
)

del blocs_retenus

print("Observations retenues :", len(revenus_long))

display(
    revenus_long.groupby(
        ["GEO_OBJECT", "FILOSOFI_MEASURE"]
    ).size()
)

500,000 lignes lues
1,000,000 lignes lues
Observations retenues : 2534


GEO_OBJECT  FILOSOFI_MEASURE
COM         MED_SL              1266
            PR_MD60             1266
REG         MED_SL                 1
            PR_MD60                1
dtype: int64

In [17]:
#Vérifier les unités et l'unicité

unites_attendues = {
    "MED_SL": "EUR_YR",
    "PR_MD60": "PT",
}

for mesure, unite in unites_attendues.items():
    selection = revenus_long[
        revenus_long["FILOSOFI_MEASURE"].eq(mesure)
    ]

    if selection.empty:
        raise ValueError(
            f"La mesure {mesure} est absente."
        )

    if not selection["UNIT_MEASURE"].eq(unite).all():
        raise ValueError(
            f"Unité inattendue pour {mesure}."
        )

    if not selection["UNIT_MULT"].eq("0").all():
        raise ValueError(
            f"Multiplicateur inattendu pour {mesure}."
        )

cles_observation = [
    "GEO_OBJECT",
    "GEO",
    "TIME_PERIOD",
    "FILOSOFI_MEASURE",
]

doublons = revenus_long[
    revenus_long.duplicated(
        subset=cles_observation,
        keep=False,
    )
]

if not doublons.empty:
    enregistrer_csv(
        doublons,
        DOSSIER_INTERIM / "doublons_source.csv",
    )

    raise ValueError(
        "Plusieurs valeurs existent pour une même "
        "mesure et un même territoire."
    )

print("Unités et unicité vérifiées.")

Unités et unicité vérifiées.


In [18]:
#Traiter les valeurs et leur statut

confidentialites_inconnues = (
    set(revenus_long["CONF_STATUS"]) - {"F", "C"}
)

statuts_inconnus = (
    set(revenus_long["OBS_STATUS"]) - {"A", "O"}
)

if confidentialites_inconnues or statuts_inconnus:
    raise ValueError(
        "Nouveaux statuts à interpréter dans la documentation : "
        f"{confidentialites_inconnues}, {statuts_inconnus}"
    )

valeur_texte = (
    revenus_long["OBS_VALUE"]
    .str.replace("\u202f", "", regex=False)
    .str.replace("\u00a0", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(",", ".", regex=False)
)

valeur_numerique = pd.to_numeric(
    valeur_texte,
    errors="coerce",
)

publication_autorisee = (
    revenus_long["CONF_STATUS"].eq("F")
    & revenus_long["OBS_STATUS"].eq("A")
)

erreurs_conversion = (
    publication_autorisee
    & valeur_texte.ne("")
    & valeur_numerique.isna()
)

if erreurs_conversion.any():
    display(revenus_long.loc[erreurs_conversion])

    raise ValueError(
        "Des valeurs publiées ne sont pas numériques."
    )

revenus_long["VALEUR"] = valeur_numerique.where(
    publication_autorisee
)

revenus_long["DISPONIBILITE"] = "MANQUANTE_SOURCE"

revenus_long.loc[
    revenus_long["CONF_STATUS"].eq("C"),
    "DISPONIBILITE",
] = "CONFIDENTIELLE"

revenus_long.loc[
    publication_autorisee
    & revenus_long["VALEUR"].notna(),
    "DISPONIBILITE",
] = "DISPONIBLE"

display(
    revenus_long.groupby(
        ["FILOSOFI_MEASURE", "DISPONIBILITE"]
    ).size()
)

enregistrer_csv(
    revenus_long,
    DOSSIER_INTERIM / "observations_revenus_idf_2023.csv",
)

FILOSOFI_MEASURE  DISPONIBILITE 
MED_SL            CONFIDENTIELLE      14
                  DISPONIBLE        1253
PR_MD60           CONFIDENTIELLE     723
                  DISPONIBLE         544
dtype: int64

observations_revenus_idf_2023.csv : 2,534 lignes, 11 colonnes


In [19]:
#Transformer les données en table communale

revenus_communes_long = revenus_long.loc[
    revenus_long["GEO_OBJECT"].eq("COM")
].copy()

valeurs_communes = revenus_communes_long.pivot(
    index="GEO",
    columns="FILOSOFI_MEASURE",
    values="VALEUR",
)

statuts_communes = revenus_communes_long.pivot(
    index="GEO",
    columns="FILOSOFI_MEASURE",
    values="DISPONIBILITE",
)

valeurs_communes = valeurs_communes.reindex(
    columns=["MED_SL", "PR_MD60"]
)

statuts_communes = statuts_communes.reindex(
    columns=["MED_SL", "PR_MD60"]
).fillna("MESURE_ABSENTE_SOURCE")

valeurs_communes = valeurs_communes.rename(
    columns={
        "MED_SL": "NIVEAU_VIE_MEDIAN_2023",
        "PR_MD60": "TAUX_PAUVRETE_60_2023",
    }
)

statuts_communes = statuts_communes.rename(
    columns={
        "MED_SL": "STATUT_NIVEAU_VIE_2023",
        "PR_MD60": "STATUT_PAUVRETE_2023",
    }
)

revenus_communes = (
    valeurs_communes
    .join(statuts_communes, validate="one_to_one")
    .rename_axis("CODGEO")
    .reset_index()
)

revenus_communes.columns.name = None

revenus_communes["CODGEO"] = (
    revenus_communes["CODGEO"].astype("string")
)

revenus_communes["MILLESIME_REVENUS"] = 2023
revenus_communes["GEOGRAPHIE_REVENUS"] = "2026-01-01"
revenus_communes["SOURCE_REVENUS"] = "INSEE_FILOSOFI_2"

assert revenus_communes["CODGEO"].is_unique

print("Communes de la source IDF :", len(revenus_communes))

display(revenus_communes.head())

Communes de la source IDF : 1266


,CODGEO,NIVEAU_VIE_MEDIAN_2023,TAUX_PAUVRETE_60_2023,STATUT_NIVEAU_VIE_2023,STATUT_PAUVRETE_2023,MILLESIME_REVENUS,GEOGRAPHIE_REVENUS,SOURCE_REVENUS
0,75056,33650.0,16.8,DISPONIBLE,DISPONIBLE,2023,2026-01-01,INSEE_FILOSOFI_2
1,77001,34840.0,<NA>,DISPONIBLE,CONFIDENTIELLE,2023,2026-01-01,INSEE_FILOSOFI_2
2,77002,29130.0,<NA>,DISPONIBLE,CONFIDENTIELLE,2023,2026-01-01,INSEE_FILOSOFI_2
3,77003,30310.0,<NA>,DISPONIBLE,CONFIDENTIELLE,2023,2026-01-01,INSEE_FILOSOFI_2
4,77004,31970.0,<NA>,DISPONIBLE,CONFIDENTIELLE,2023,2026-01-01,INSEE_FILOSOFI_2


In [20]:
#Vérifier les valeurs du niveau de vie

mediane = revenus_communes[
    "NIVEAU_VIE_MEDIAN_2023"
].dropna()

pauvrete = revenus_communes[
    "TAUX_PAUVRETE_60_2023"
].dropna()

if mediane.empty:
    raise ValueError(
        "Aucun niveau de vie médian disponible."
    )

if not np.isfinite(mediane.to_numpy(dtype=float)).all():
    raise ValueError("Niveau de vie non fini.")

if not np.isfinite(pauvrete.to_numpy(dtype=float)).all():
    raise ValueError("Taux de pauvreté non fini.")

if not mediane.ge(0).all():
    raise ValueError("Niveau de vie médian négatif.")

if not pauvrete.between(0, 100).all():
    raise ValueError(
        "Taux de pauvreté hors de l'intervalle 0–100."
    )

display(
    revenus_communes[
        [
            "NIVEAU_VIE_MEDIAN_2023",
            "TAUX_PAUVRETE_60_2023",
        ]
    ].describe()
)

,NIVEAU_VIE_MEDIAN_2023,TAUX_PAUVRETE_60_2023
count,1252.0,543.0
mean,30568.634185,13.38582
std,4888.699081,8.109789
min,16470.0,3.0
25%,27657.5,8.0
50%,30295.0,10.4
75%,33120.0,17.0
max,56030.0,45.4


In [21]:
#Contrôler la géographie

codes_source = set(revenus_communes["CODGEO"])

codes_sans_source = sorted(
    codes_profil - codes_source
)

codes_source_hors_profil = sorted(
    codes_source - codes_profil
)

diagnostic_sans_source = profil_j9.loc[
    profil_j9["CODGEO"].isin(codes_sans_source)
].copy()

diagnostic_hors_profil = revenus_communes.loc[
    revenus_communes["CODGEO"].isin(
        codes_source_hors_profil
    )
].copy()

print(
    "Communes du profil sans ligne source :",
    len(codes_sans_source),
)

print(
    "Communes de la source absentes du profil :",
    len(codes_source_hors_profil),
)

display(diagnostic_sans_source.head(20))
display(diagnostic_hors_profil.head(20))

enregistrer_csv(
    diagnostic_sans_source,
    DOSSIER_INTERIM / "diagnostic_communes_sans_source.csv",
)

enregistrer_csv(
    diagnostic_hors_profil,
    DOSSIER_INTERIM / "diagnostic_codes_source_hors_profil.csv",
)

codes_arrondissements_paris = {
    f"751{numero:02d}"
    for numero in range(1, 21)
}

if codes_profil.intersection(codes_arrondissements_paris):
    raise ValueError(
        "Le profil contient des arrondissements parisiens. "
        "Cette version du J10 utilise les communes COM ; "
        "il faut adapter le périmètre."
    )

if "93059" in codes_profil:
    raise ValueError(
        "Le profil contient encore Pierrefitte séparément. "
        "Il faut réconcilier sa géographie avec la source 2026."
    )

Communes du profil sans ligne source : 0
Communes de la source absentes du profil : 0


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,POP_18_24,POP_25_39,POP_40_54,POP_55_64,POP_65_79,POP_80_PLUS,POP_FEMMES,POP_HOMMES,POP_F_0_2,POP_F_3_5,...,PART_ACTIFS_OCCUPES_RESIDENTS_ARTISANS_COMMERCANTS_CHEFS_PCT,PART_EMPLOIS_LT_CADRES_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_CADRES_PCT,PART_EMPLOIS_LT_PROF_INTERMEDIAIRES_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_PROF_INTERMEDIAIRES_PCT,PART_EMPLOIS_LT_EMPLOYES_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_EMPLOYES_PCT,PART_EMPLOIS_LT_OUVRIERS_PCT,PART_ACTIFS_OCCUPES_RESIDENTS_OUVRIERS_PCT,PART_EMPLOIS_LT_AGRICULTURE_PCT,PART_EMPLOIS_LT_INDUSTRIE_PCT,PART_EMPLOIS_LT_CONSTRUCTION_PCT,PART_EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES_PCT,PART_EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL_PCT,EMPLOIS_TERTIAIRES_LT,PART_EMPLOIS_TERTIAIRES_LT_PCT,EMPLOIS_15P_LT_POUR_1000_HAB,INDICE_DENSITE_EMPLOIS_IDF_BASE100,NB_RESTAURATION_RAPIDE_1000_EMPLOIS,INDICE_DENSITE_RAPIDE_PAR_EMPLOI_IDF_BASE100,NB_RESTAURANTS_TOTAL_1000_EMPLOIS,NB_LIEUX_TRANSPORT,NB_LIEUX_BUS,NB_STATIONS_METRO,NB_STATIONS_TRAMWAY,NB_GARES_RER,NB_GARES_TRAIN,NB_STATIONS_FUNICULAIRE,NB_LIEUX_FERRES,NB_GARES_STATIONS_TRANSPORT_LOURD,NB_LIEUX_MULTIMODAUX,NB_POLES_MULTIMODAUX_LOURDS,NB_LIGNES_TRANSPORT,NB_MODES_TRANSPORT_PRESENTS,NB_OPERATEURS_TRANSPORT,NB_LIGNES_BUS,NB_LIGNES_METRO,NB_LIGNES_TRAMWAY,NB_LIGNES_RER,NB_LIGNES_TRAIN,NB_LIGNES_FUNICULAIRE,NB_LIGNES_AUTRES,NB_LIGNES_FERREES,NB_LIGNES_TRANSPORT_LOURD,A_BUS,A_METRO,A_TRAMWAY,A_RER,A_TRAIN,A_TRANSPORT_LOURD,NB_LIEUX_TRANSPORT_10000_HAB,NB_LIGNES_TRANSPORT_10000_HAB,NB_LIGNES_FERREES_10000_HAB,NB_GARES_STATIONS_LOURDES_10000_HAB,NB_POLES_MULTIMODAUX_10000_HAB,INDICE_LIEUX_TRANSPORT_IDF_BASE100,INDICE_LIGNES_FERREES_IDF_BASE100,NB_LIGNES_TRANSPORT_1000_EMPLOIS,NB_GARES_STATIONS_LOURDES_1000_EMPLOIS,CLASSE_DESSERTE_TRANSPORT


,CODGEO,NIVEAU_VIE_MEDIAN_2023,TAUX_PAUVRETE_60_2023,STATUT_NIVEAU_VIE_2023,STATUT_PAUVRETE_2023,MILLESIME_REVENUS,GEOGRAPHIE_REVENUS,SOURCE_REVENUS


diagnostic_communes_sans_source.csv : 0 lignes, 403 colonnes
diagnostic_codes_source_hors_profil.csv : 0 lignes, 8 colonnes


In [22]:
#Charger les références régionales officielles

reference_region = revenus_long.loc[
    revenus_long["GEO_OBJECT"].eq("REG")
    & revenus_long["GEO"].eq("11")
].copy()

if (
    len(reference_region) != 2
    or set(reference_region["FILOSOFI_MEASURE"])
    != MESURES_RETENUES
):
    raise ValueError(
        "Référence régionale absente ou incomplète."
    )

if not reference_region["DISPONIBILITE"].eq(
    "DISPONIBLE"
).all():
    raise ValueError(
        "Une référence régionale n'est pas disponible."
    )

references = reference_region.set_index(
    "FILOSOFI_MEASURE"
)["VALEUR"]

MEDIANE_IDF_2023 = float(references["MED_SL"])
PAUVRETE_IDF_2023 = float(references["PR_MD60"])

if MEDIANE_IDF_2023 <= 0:
    raise ValueError("Médiane régionale invalide.")

if not 0 <= PAUVRETE_IDF_2023 <= 100:
    raise ValueError("Taux régional invalide.")

print(
    "Niveau de vie médian IDF :",
    MEDIANE_IDF_2023,
    "euros annuels",
)

print(
    "Taux de pauvreté IDF :",
    PAUVRETE_IDF_2023,
    "%",
)

enregistrer_csv(
    reference_region,
    DOSSIER_INTERIM / "reference_revenus_idf_2023.csv",
)

Niveau de vie médian IDF : 28210.0 euros annuels
Taux de pauvreté IDF : 17.3 %
reference_revenus_idf_2023.csv : 2 lignes, 11 colonnes


In [23]:
#Joindre les revenus au profil J9

nouvelles_colonnes = [
    colonne
    for colonne in revenus_communes.columns
    if colonne != "CODGEO"
]

collisions = set(nouvelles_colonnes).intersection(
    profil_j9.columns
)

if collisions:
    raise ValueError(
        "Ces colonnes sont déjà présentes dans le profil J9 : "
        f"{sorted(collisions)}. "
        "Compare leur provenance avant de les remplacer."
    )

profil_j10 = profil_j9.merge(
    revenus_communes,
    on="CODGEO",
    how="left",
    validate="one_to_one",
    indicator="_JOINTURE_REVENUS",
)

profil_j10["REVENU_SOURCE_APPARIEE_2023"] = (
    profil_j10["_JOINTURE_REVENUS"]
    .eq("both")
    .astype(int)
)

for colonne in [
    "STATUT_NIVEAU_VIE_2023",
    "STATUT_PAUVRETE_2023",
]:
    profil_j10[colonne] = (
        profil_j10[colonne]
        .fillna("COMMUNE_ABSENTE_SOURCE")
    )

profil_j10 = profil_j10.drop(
    columns="_JOINTURE_REVENUS"
)

print("Communes avant :", len(profil_j9))
print("Communes après :", len(profil_j10))

assert len(profil_j10) == len(profil_j9)
assert profil_j10["CODGEO"].is_unique

Communes avant : 1266
Communes après : 1266


In [24]:
#Mesurer la couverture réelle

indicateurs_couverture = [
    (
        "NIVEAU_VIE_MEDIAN_2023",
        "STATUT_NIVEAU_VIE_2023",
    ),
    (
        "TAUX_PAUVRETE_60_2023",
        "STATUT_PAUVRETE_2023",
    ),
]

lignes_couverture = []

for indicateur, colonne_statut in indicateurs_couverture:
    for statut, nombre in (
        profil_j10[colonne_statut]
        .value_counts(dropna=False)
        .items()
    ):
        lignes_couverture.append({
            "INDICATEUR": indicateur,
            "STATUT": statut,
            "NB_COMMUNES": int(nombre),
            "PART_COMMUNES_PCT": (
                100 * nombre / len(profil_j10)
            ),
        })

controle_couverture = pd.DataFrame(
    lignes_couverture
)

display(controle_couverture)

enregistrer_csv(
    controle_couverture,
    DOSSIER_INTERIM / "controle_couverture_revenus.csv",
)

,INDICATEUR,STATUT,NB_COMMUNES,PART_COMMUNES_PCT
0,NIVEAU_VIE_MEDIAN_2023,DISPONIBLE,1252,98.894155
1,NIVEAU_VIE_MEDIAN_2023,CONFIDENTIELLE,14,1.105845
2,TAUX_PAUVRETE_60_2023,CONFIDENTIELLE,723,57.109005
3,TAUX_PAUVRETE_60_2023,DISPONIBLE,543,42.890995


controle_couverture_revenus.csv : 4 lignes, 4 colonnes


In [25]:
#Calculer les comparaison avec l'IDF

profil_j10[
    "INDICE_NIVEAU_VIE_MEDIAN_IDF_BASE100_2023"
] = (
    profil_j10["NIVEAU_VIE_MEDIAN_2023"]
    / MEDIANE_IDF_2023
    * 100
)

profil_j10[
    "ECART_TAUX_PAUVRETE_IDF_POINTS_2023"
] = (
    profil_j10["TAUX_PAUVRETE_60_2023"]
    - PAUVRETE_IDF_2023
)

profil_j10[
    "NIVEAU_VIE_DISPONIBLE_2023"
] = (
    profil_j10["NIVEAU_VIE_MEDIAN_2023"]
    .notna()
    .astype(int)
)

profil_j10[
    "PAUVRETE_DISPONIBLE_2023"
] = (
    profil_j10["TAUX_PAUVRETE_60_2023"]
    .notna()
    .astype(int)
)

In [26]:
#Examiner les communes

colonnes_nom_possibles = [
    "LIBELLE",
    "NOM_COMMUNE",
    "LIBGEO",
    "NOM_COM",
]

COLONNE_NOM = next(
    (
        colonne
        for colonne in colonnes_nom_possibles
        if colonne in profil_j10.columns
    ),
    None,
)

colonnes_affichage = [
    "CODGEO",
    COLONNE_NOM,
    "NIVEAU_VIE_MEDIAN_2023",
    "TAUX_PAUVRETE_60_2023",
    "INDICE_NIVEAU_VIE_MEDIAN_IDF_BASE100_2023",
    "STATUT_NIVEAU_VIE_2023",
    "STATUT_PAUVRETE_2023",
]

colonnes_affichage = [
    colonne
    for colonne in colonnes_affichage
    if colonne is not None
]

print("Niveaux de vie médians les plus élevés :")

display(
    profil_j10.loc[
        profil_j10["NIVEAU_VIE_MEDIAN_2023"].notna(),
        colonnes_affichage,
    ]
    .sort_values(
        "NIVEAU_VIE_MEDIAN_2023",
        ascending=False,
    )
    .head(20)
)

print("Niveaux de vie médians les plus faibles :")

display(
    profil_j10.loc[
        profil_j10["NIVEAU_VIE_MEDIAN_2023"].notna(),
        colonnes_affichage,
    ]
    .sort_values("NIVEAU_VIE_MEDIAN_2023")
    .head(20)
)

Niveaux de vie médians les plus élevés :


,CODGEO,NOM_COMMUNE,NIVEAU_VIE_MEDIAN_2023,TAUX_PAUVRETE_60_2023,INDICE_NIVEAU_VIE_MEDIAN_IDF_BASE100_2023,STATUT_NIVEAU_VIE_2023,STATUT_PAUVRETE_2023
985,92051,Neuilly-sur-Seine,56030.0,7.5,198.617512,DISPONIBLE,DISPONIBLE
729,78571,Saint-Nom-la-Bretèche,50940.0,3.0,180.574264,DISPONIBLE,DISPONIBLE
756,78650,Le Vésinet,49120.0,6.0,174.122652,DISPONIBLE,DISPONIBLE
590,78233,Feucherolles,47730.0,3.0,169.195321,DISPONIBLE,DISPONIBLE
562,78152,Chavenay,47470.0,<NA>,168.273662,DISPONIBLE,CONFIDENTIELLE
994,92076,Vaucresson,47390.0,7.0,167.990074,DISPONIBLE,DISPONIBLE
511,78007,Aigremont,46770.0,<NA>,165.792272,DISPONIBLE,CONFIDENTIELLE
981,92047,Marnes-la-Coquette,46710.0,<NA>,165.579582,DISPONIBLE,CONFIDENTIELLE
579,78196,Davron,46040.0,<NA>,163.204537,DISPONIBLE,CONFIDENTIELLE
989,92064,Saint-Cloud,45740.0,7.7,162.141085,DISPONIBLE,DISPONIBLE


Niveaux de vie médians les plus faibles :


,CODGEO,NOM_COMMUNE,NIVEAU_VIE_MEDIAN_2023,TAUX_PAUVRETE_60_2023,INDICE_NIVEAU_VIE_MEDIAN_IDF_BASE100_2023,STATUT_NIVEAU_VIE_2023,STATUT_PAUVRETE_2023
849,91286,Grigny,16470.0,44.9,58.383552,DISPONIBLE,DISPONIBLE
1004,93014,Clichy-sous-Bois,16520.0,45.4,58.560794,DISPONIBLE,DISPONIBLE
1006,93027,La Courneuve,16820.0,43.0,59.624247,DISPONIBLE,DISPONIBLE
997,93001,Aubervilliers,17210.0,42.0,61.006735,DISPONIBLE,DISPONIBLE
1157,95268,Garges-lès-Gonesse,17320.0,40.8,61.396668,DISPONIBLE,DISPONIBLE
1030,93072,Stains,17670.0,39.9,62.637363,DISPONIBLE,DISPONIBLE
1263,95680,Villiers-le-Bel,17750.0,39.4,62.92095,DISPONIBLE,DISPONIBLE
1001,93008,Bobigny,18050.0,37.9,63.984403,DISPONIBLE,DISPONIBLE
1027,93066,Saint-Denis,18130.0,38.6,64.26799,DISPONIBLE,DISPONIBLE
1240,95585,Sarcelles,18180.0,37.3,64.445232,DISPONIBLE,DISPONIBLE


In [27]:
#Croiser revenus, emploi et concurrence

colonnes_analyse = [
    "CODGEO",
    COLONNE_NOM,
    "NIVEAU_VIE_MEDIAN_2023",
    "TAUX_PAUVRETE_60_2023",
    "EMPLOIS_15P_LT",
    "NB_RESTAURANTS_TOTAL",
    "NB_RESTAURATION_RAPIDE",
    "NB_RESTAURATION_RAPIDE_1000_EMPLOIS",
    "NB_LIGNES_TRANSPORT_LOURD",
]

colonnes_analyse = [
    colonne
    for colonne in colonnes_analyse
    if colonne is not None
    and colonne in profil_j10.columns
]

analyse_communes = profil_j10[
    colonnes_analyse
].copy()

display(analyse_communes.head(20))

enregistrer_csv(
    analyse_communes,
    DOSSIER_INTERIM / "lecture_croisee_revenus_emploi_concurrence.csv",
)

,CODGEO,NOM_COMMUNE,NIVEAU_VIE_MEDIAN_2023,TAUX_PAUVRETE_60_2023,EMPLOIS_15P_LT,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_RAPIDE_1000_EMPLOIS,NB_LIGNES_TRANSPORT_LOURD
0,75056,Paris,33650.0,16.8,1.810931e+06,20911,8329,4.599291,42
1,77001,Achères-la-Forêt,34840.0,<NA>,1.926234e+02,1,0,0.000000,0
2,77002,Amillis,29130.0,<NA>,2.560721e+02,2,2,7.810300,0
3,77003,Amponville,30310.0,<NA>,3.493564e+01,1,0,0.000000,0
4,77004,Andrezel,31970.0,<NA>,3.556987e+01,0,0,0.000000,0
5,77005,Annet-sur-Marne,32970.0,5.0,5.357558e+02,6,2,3.733044,0
6,77006,Arbonne-la-Forêt,34140.0,<NA>,2.579764e+02,5,2,7.752646,0
7,77007,Argentières,32250.0,<NA>,2.920120e+01,0,0,0.000000,0
8,77008,Armentières-en-Brie,26210.0,<NA>,6.468415e+01,0,0,0.000000,0
9,77009,Arville,27070.0,<NA>,3.083824e+01,0,0,0.000000,0


lecture_croisee_revenus_emploi_concurrence.csv : 1,266 lignes, 9 colonnes


In [28]:
#Enregistrer les résultats

enregistrer_csv(
    revenus_communes,
    DOSSIER_INTERIM / "revenus_communes_idf_2023.csv",
)

enregistrer_csv(
    profil_j10,
    FICHIER_PROFIL_J10,
)

print()
print("Profil J10 enregistré.")
print("Communes :", len(profil_j10))
print("Colonnes :", len(profil_j10.columns))

revenus_communes_idf_2023.csv : 1,266 lignes, 8 colonnes
profil_communes_idf_j10.csv : 1,266 lignes, 415 colonnes

Profil J10 enregistré.
Communes : 1266
Colonnes : 415


In [29]:
#Enregistrer la tracabilité

metadonnees_j10 = {
    "notebook": "10_revenus_niveau_vie_idf.ipynb",
    "source": "Insee - Filosofi 2",
    "url_source": "https://www.insee.fr/fr/statistiques/8984752",
    "fichier_source": str(
        FICHIER_SOURCE.relative_to(RACINE)
    ),
    "sha256_source": empreinte_sha256(FICHIER_SOURCE),
    "membre_donnees": MEMBRE_DATA,
    "millesime_statistique": 2023,
    "geographie_source": "2026-01-01",
    "date_telechargement": None,
    "date_traitement_utc": datetime.now(
        timezone.utc
    ).isoformat(timespec="seconds"),
    "indicateurs": {
        "MED_SL": "Niveau de vie médian annuel",
        "PR_MD60": "Taux de pauvreté au seuil de 60 %",
    },
    "reference_idf": {
        "niveau_vie_median": MEDIANE_IDF_2023,
        "taux_pauvrete": PAUVRETE_IDF_2023,
    },
    "nb_communes_profil": int(len(profil_j10)),
    "nb_communes_sans_ligne_source": int(
        len(diagnostic_sans_source)
    ),
    "nb_codes_source_hors_profil": int(
        len(diagnostic_hors_profil)
    ),
    "regles": [
        "Valeurs confidentielles conservées comme manquantes",
        "Aucune agrégation de médianes communales",
        "Références régionales issues de la source officielle",
        "Indicateurs antérieurs du profil préservés",
    ],
}

FICHIER_METADATA = (
    DOSSIER_INTERIM / "metadata_j10.json"
)

FICHIER_METADATA.write_text(
    json.dumps(
        metadonnees_j10,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Métadonnées enregistrées.")

Métadonnées enregistrées.


## Bilan du J10

Le profil communal J9 a été enrichi avec :

- le niveau de vie médian en 2023 ;
- le taux de pauvreté à 60 % en 2023 ;
- les statuts de disponibilité de ces indicateurs ;
- un indice de niveau de vie médian, référence IDF = 100 ;
- l'écart du taux de pauvreté avec la région.

Les valeurs confidentielles ou manquantes n'ont pas été imputées.

Les indicateurs antérieurs de population, d'emploi, de concurrence
et de transport ont été conservés.

Les revenus décrivent le contexte économique résidentiel.
Ils ne mesurent pas directement la dépense de restauration
ni la clientèle présente en journée.

Les éventuelles communes non appariées figurent dans les diagnostics.